In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
inflection = pd.read_csv('data/extrema_points_results.csv')

In [ ]:
inflection = pd.read_csv('data/extrema_points_results_additional.csv')

Update: re-derive the inflection points using the new region types obtained by applying GMM to the GAM-significant regions

In [ ]:
def manipulate_roi(roi):
    roi = roi.replace('_ROI', '')
    roi = roi.replace('.', '-')
    if roi.startswith('R'):
        return 'rh_' + roi
    elif roi.startswith('L'):
        return 'lh_' + roi
    return roi

In [ ]:
face_sig = pd.read_csv('data/GMM_gam_sig_face.csv')
place_sig = pd.read_csv('data/GMM_gam_sig_place.csv')
body_sig = pd.read_csv('data/GMM_gam_sig_body.csv')
tool_sig = pd.read_csv('data/GMM_gam_sig_tool.csv')

In [ ]:
# face_sig = pd.read_csv('data/WM_Face_2bk_fdr_results_with_rank.csv')
# place_sig = pd.read_csv('data/WM_Place_2bk_fdr_results_with_rank.csv')

In [ ]:
inflection['Region'] = inflection['Region'].apply(manipulate_roi)

In [ ]:
inflection

In [ ]:
# Operate on the Region column of the inflection table

In [ ]:
From the inflection table, take rows with Dataset == WM_Face_2bk to form inflection_face, then add the gmm_nonlinear column from face_sig, matching the inflection Region column to the face_sig label column.
From the inflection table, take rows with Dataset == WM_Place_2bk to form inflection_place, then add the gmm_nonlinear column from place_sig, matching the inflection Region column to the place_sig label column.

In [ ]:
# To match, normalize the Region column by removing the "_ROI" suffix and build a mapping dict
face_sig_mapping = dict(zip(face_sig['label'], face_sig['gmm_nonlinear']))
place_sig_mapping = dict(zip(place_sig['label'], place_sig['gmm_nonlinear']))
body_sig_mapping = dict(zip(body_sig['label'], body_sig['gmm_nonlinear']))
tool_sig_mapping = dict(zip(tool_sig['label'], tool_sig['gmm_nonlinear']))

# Process inflection_face
inflection_face = inflection[inflection['Dataset'] == 'WM_Face_2bk'].copy()
# Remove the "_ROI" suffix from the Region column to match the face_sig format
inflection_face['Region_clean'] = inflection_face['Region'].str.replace('_ROI', '')
# Add the gmm_nonlinear column
inflection_face['gmm_nonlinear'] = inflection_face['Region_clean'].map(face_sig_mapping)

# Process inflection_place
inflection_place = inflection[inflection['Dataset'] == 'WM_Place_2bk'].copy()
# Remove the "_ROI" suffix from the Region column to match the place_sig format
inflection_place['Region_clean'] = inflection_place['Region'].str.replace('_ROI', '')
# Add the gmm_nonlinear column
inflection_place['gmm_nonlinear'] = inflection_place['Region_clean'].map(place_sig_mapping)

# Process inflection_body
inflection_body = inflection[inflection['Dataset'] == 'WM_Body_2bk'].copy()
# Remove the "_ROI" suffix from the Region column to match the body_sig format
inflection_body['Region_clean'] = inflection_body['Region'].str.replace('_ROI', '')
# Add the gmm_nonlinear column
inflection_body['gmm_nonlinear'] = inflection_body['Region_clean'].map(body_sig_mapping)

# Process inflection_tool
inflection_tool = inflection[inflection['Dataset'] == 'WM_Tool_2bk'].copy()
# Remove the "_ROI" suffix from the Region column to match the tool_sig format
inflection_tool['Region_clean'] = inflection_tool['Region'].str.replace('_ROI', '')
# Add the gmm_nonlinear column
inflection_tool['gmm_nonlinear'] = inflection_tool['Region_clean'].map(tool_sig_mapping)

# Drop the helper column
inflection_face = inflection_face.drop('Region_clean', axis=1)
inflection_place = inflection_place.drop('Region_clean', axis=1)
inflection_body = inflection_body.drop('Region_clean', axis=1)
inflection_tool = inflection_tool.drop('Region_clean', axis=1)

# Keep only successfully matched rows (gmm_nonlinear not NaN)
inflection_face = inflection_face.dropna(subset=['gmm_nonlinear'])
inflection_place = inflection_place.dropna(subset=['gmm_nonlinear'])
inflection_body = inflection_body.dropna(subset=['gmm_nonlinear'])
inflection_tool = inflection_tool.dropna(subset=['gmm_nonlinear'])

print(f"inflection_face rows after matching: {len(inflection_face)}")
print(f"inflection_place rows after matching: {len(inflection_place)}")
print(f"inflection_body rows after matching: {len(inflection_body)}")
print(f"inflection_tool rows after matching: {len(inflection_tool)}")

In [ ]:
inflection_face

In [ ]:
# inflection_face = inflection[(inflection['Dataset'] == 'WM_Face_2bk') & (inflection['Region'].isin(face_sig['ROI']))]
# inflection_place = inflection[(inflection['Dataset'] == 'WM_Place_2bk') & (inflection['Region'].isin(place_sig['ROI']))]

In [ ]:
inflection_face_ConcaveIncrease = inflection_face[(inflection_face['Mean_First_Derivative'] > 0) & (inflection_face['Mean_Second_Derivative'] < 0) & (inflection_face['gmm_nonlinear'] == 1)]
inflection_place_ConcaveIncrease = inflection_place[(inflection_place['Mean_First_Derivative'] > 0) & (inflection_place['Mean_Second_Derivative'] < 0) & (inflection_place['gmm_nonlinear'] == 1)]
inflection_face_ConvexIncrease = inflection_face[(inflection_face['Mean_First_Derivative'] > 0) & (inflection_face['Mean_Second_Derivative'] > 0) & (inflection_face['gmm_nonlinear'] == 1)]
inflection_place_ConvexIncrease = inflection_place[(inflection_place['Mean_First_Derivative'] > 0) & (inflection_place['Mean_Second_Derivative'] > 0) & (inflection_place['gmm_nonlinear'] == 1)]
inflection_face_ConcaveDecrease = inflection_face[(inflection_face['Mean_First_Derivative'] < 0) & (inflection_face['Mean_Second_Derivative'] < 0) & (inflection_face['gmm_nonlinear'] == 1)]
inflection_place_ConcaveDecrease = inflection_place[(inflection_place['Mean_First_Derivative'] < 0) & (inflection_place['Mean_Second_Derivative'] < 0) & (inflection_place['gmm_nonlinear'] == 1)]
inflection_face_ConvexDecrease = inflection_face[(inflection_face['Mean_First_Derivative'] < 0) & (inflection_face['Mean_Second_Derivative'] > 0) & (inflection_face['gmm_nonlinear'] == 1)]
inflection_place_ConvexDecrease = inflection_place[(inflection_place['Mean_First_Derivative'] < 0) & (inflection_place['Mean_Second_Derivative'] > 0) & (inflection_place['gmm_nonlinear'] == 1)]
inflection_body_ConvexDecrease = inflection_body[(inflection_body['Mean_First_Derivative'] < 0) & (inflection_body['Mean_Second_Derivative'] > 0) & (inflection_body['gmm_nonlinear'] == 1)]
inflection_tool_ConvexDecrease = inflection_tool[(inflection_tool['Mean_First_Derivative'] < 0) & (inflection_tool['Mean_Second_Derivative'] > 0) & (inflection_tool['gmm_nonlinear'] == 1)]
inflection_body_ConcaveIncrease = inflection_body[(inflection_body['Mean_First_Derivative'] > 0) & (inflection_body['Mean_Second_Derivative'] < 0) & (inflection_body['gmm_nonlinear'] == 1)]
inflection_tool_ConcaveIncrease = inflection_tool[(inflection_tool['Mean_First_Derivative'] > 0) & (inflection_tool['Mean_Second_Derivative'] < 0) & (inflection_tool['gmm_nonlinear'] == 1)]
inflection_body_ConvexIncrease = inflection_body[(inflection_body['Mean_First_Derivative'] > 0) & (inflection_body['Mean_Second_Derivative'] > 0) & (inflection_body['gmm_nonlinear'] == 1)]
inflection_tool_ConvexIncrease = inflection_tool[(inflection_tool['Mean_First_Derivative'] > 0) & (inflection_tool['Mean_Second_Derivative'] > 0) & (inflection_tool['gmm_nonlinear'] == 1)]
inflection_body_ConcaveDecrease = inflection_body[(inflection_body['Mean_First_Derivative'] < 0) & (inflection_body['Mean_Second_Derivative'] < 0) & (inflection_body['gmm_nonlinear'] == 1)]
inflection_tool_ConcaveDecrease = inflection_tool[(inflection_tool['Mean_First_Derivative'] < 0) & (inflection_tool['Mean_Second_Derivative'] < 0) & (inflection_tool['gmm_nonlinear'] == 1)]


In [ ]:
inflection_face_ConcaveDecrease.dropna(inplace = True)
inflection_face_ConvexDecrease.dropna(inplace = True)
inflection_place_ConvexDecrease.dropna(inplace = True)
inflection_place_ConcaveDecrease.dropna(inplace = True)
inflection_face_ConcaveIncrease.dropna(inplace = True)
inflection_face_ConvexIncrease.dropna(inplace = True)
inflection_place_ConvexIncrease.dropna(inplace = True)
inflection_place_ConcaveIncrease.dropna(inplace = True)
inflection_body_ConvexDecrease.dropna(inplace = True)
inflection_body_ConcaveIncrease.dropna(inplace = True)
inflection_body_ConvexIncrease.dropna(inplace = True)
inflection_body_ConcaveDecrease.dropna(inplace = True)
inflection_tool_ConcaveIncrease.dropna(inplace = True)
inflection_tool_ConvexIncrease.dropna(inplace = True)
inflection_tool_ConcaveDecrease.dropna(inplace = True)
inflection_tool_ConvexDecrease.dropna(inplace = True)

In [ ]:
print("=== Dataset Sizes ===")
print(f"inflection_face_ConcaveDecrease: {len(inflection_face_ConcaveDecrease)} rows")
print(f"inflection_face_ConvexDecrease: {len(inflection_face_ConvexDecrease)} rows")
print(f"inflection_place_ConvexDecrease: {len(inflection_place_ConvexDecrease)} rows")
print(f"inflection_place_ConcaveDecrease: {len(inflection_place_ConcaveDecrease)} rows")
print(f"inflection_face_ConcaveIncrease: {len(inflection_face_ConcaveIncrease)} rows")
print(f"inflection_face_ConvexIncrease: {len(inflection_face_ConvexIncrease)} rows")
print(f"inflection_place_ConvexIncrease: {len(inflection_place_ConvexIncrease)} rows")
print(f"inflection_place_ConcaveIncrease: {len(inflection_place_ConcaveIncrease)} rows")
print(f"inflection_body_ConvexDecrease: {len(inflection_body_ConvexDecrease)} rows")
print(f"inflection_tool_ConvexDecrease: {len(inflection_tool_ConvexDecrease)} rows")
print(f"inflection_body_ConcaveIncrease: {len(inflection_body_ConcaveIncrease)} rows")
print(f"inflection_tool_ConcaveIncrease: {len(inflection_tool_ConcaveIncrease)} rows")
print(f"inflection_body_ConvexIncrease: {len(inflection_body_ConvexIncrease)} rows")
print(f"inflection_tool_ConvexIncrease: {len(inflection_tool_ConvexIncrease)} rows")
print(f"inflection_body_ConcaveDecrease: {len(inflection_body_ConcaveDecrease)} rows")
print(f"inflection_tool_ConcaveDecrease: {len(inflection_tool_ConcaveDecrease)} rows")

print("\n=== Summary by Category ===")
print(f"Face datasets total: {len(inflection_face_ConcaveDecrease) + len(inflection_face_ConvexDecrease) + len(inflection_face_ConcaveIncrease) + len(inflection_face_ConvexIncrease)} rows")
print(f"Place datasets total: {len(inflection_place_ConvexDecrease) + len(inflection_place_ConcaveDecrease) + len(inflection_place_ConvexIncrease) + len(inflection_place_ConcaveIncrease)} rows")
print(f"Concave datasets total: {len(inflection_face_ConcaveDecrease) + len(inflection_face_ConcaveIncrease) + len(inflection_place_ConcaveDecrease) + len(inflection_place_ConcaveIncrease) + len(inflection_body_ConcaveDecrease) + len(inflection_body_ConcaveIncrease) + len(inflection_tool_ConcaveDecrease) + len(inflection_tool_ConcaveIncrease)} rows")
print(f"Convex datasets total: {len(inflection_face_ConvexDecrease) + len(inflection_face_ConvexIncrease) + len(inflection_place_ConvexDecrease) + len(inflection_place_ConvexIncrease) + len(inflection_body_ConvexDecrease) + len(inflection_body_ConvexIncrease) + len(inflection_tool_ConvexDecrease) + len(inflection_tool_ConvexIncrease)} rows")


In [ ]:
inflection_face_Convex = inflection_face[inflection_face['Mean_Second_Derivative'] > 0]
inflection_place_Concave = inflection_place[inflection_place['Mean_Second_Derivative'] < 0]

In [ ]:
inflection_face_Convex.dropna(inplace=True)
inflection_place_Concave.dropna(inplace=True)

In [ ]:
inflection_face_Concave

In [ ]:
inflection_place_Convex

In [ ]:
inflection_face_Convex

In [ ]:
inflection_place_Concave

In [ ]:
inflection_face_Concave = inflection_face[inflection_face['Mean_Second_Derivative'] < 0 & (inflection_face['gmm_nonlinear'] == 1)]
inflection_face_Convex = inflection_face[inflection_face['Mean_Second_Derivative'] > 0 & (inflection_face['gmm_nonlinear'] == 1)]
inflection_place_Concave = inflection_place[inflection_place['Mean_Second_Derivative'] < 0 & (inflection_place['gmm_nonlinear'] == 1)]
inflection_place_Convex = inflection_place[inflection_place['Mean_Second_Derivative'] > 0 & (inflection_place['gmm_nonlinear'] == 1)]
inflection_body_Convex = inflection_body[inflection_body['Mean_Second_Derivative'] > 0 & (inflection_body['gmm_nonlinear'] == 1)]
inflection_body_Concave = inflection_body[inflection_body['Mean_Second_Derivative'] < 0 & (inflection_body['gmm_nonlinear'] == 1)]
inflection_tool_Convex = inflection_tool[inflection_tool['Mean_Second_Derivative'] > 0 & (inflection_tool['gmm_nonlinear'] == 1)]
inflection_tool_Concave = inflection_tool[inflection_tool['Mean_Second_Derivative'] < 0 & (inflection_tool['gmm_nonlinear'] == 1)]

In [ ]:
inflection_face_Concave.dropna(inplace=True)
inflection_place_Convex.dropna(inplace=True)

In [ ]:
inflection_face_ConcaveIncrease

In [ ]:
inflection_place_ConvexIncrease

## All Concave versus All Convex
`Without distinguishing the two groups`

## Concave Increase & Convex Increase
## Face versus Place

In [ ]:
import numpy as np
from scipy import stats
import pandas as pd

def permutation_test(group1, group2, n_permutations=10000):
    """
    Permutation test for comparing means of two groups
    """
    # Compute the observed difference
    observed_diff = np.mean(group1) - np.mean(group2)
    
    # Pool the data
    combined = np.concatenate([group1, group2])
    n1, n2 = len(group1), len(group2)
    
    # Run the permutation
    permuted_diffs = []
    for _ in range(n_permutations):
        # Randomly shuffle the data
        np.random.shuffle(combined)
        
        # Re-split into groups
        perm_group1 = combined[:n1]
        perm_group2 = combined[n1:]
        
        # Compute the permuted difference
        perm_diff = np.mean(perm_group1) - np.mean(perm_group2)
        permuted_diffs.append(perm_diff)
    
    # Compute the p-value
    permuted_diffs = np.array(permuted_diffs)
    p_value = np.mean(np.abs(permuted_diffs) >= np.abs(observed_diff))
    
    return observed_diff, p_value, permuted_diffs

In [ ]:
# Extract the Extrema_Points data
face_extrema = inflection_face_ConcaveIncrease['Extrema_Points']
place_extrema = inflection_place_ConvexIncrease['Extrema_Points']

print("=== Descriptive statistics ===")
print(f"Face (ConcaveIncrease) - n: {len(face_extrema)}")
print(f"  mean: {np.mean(face_extrema):.4f}")
print(f"  SD: {np.std(face_extrema, ddof=1):.4f}")
print(f"  median: {np.median(face_extrema):.4f}")

print(f"\nPlace (ConvexIncrease) - n: {len(place_extrema)}")
print(f"  mean: {np.mean(place_extrema):.4f}")
print(f"  SD: {np.std(place_extrema, ddof=1):.4f}")
print(f"  median: {np.median(place_extrema):.4f}")

print("\n=== Statistical test results ===")

# 1. Permutation Test
print("1. Permutation Test:")
observed_diff, perm_p_value, perm_diffs = permutation_test(face_extrema, place_extrema)
print(f"   observed mean difference: {observed_diff:.4f}")
print(f"   p-value: {perm_p_value:.4f}")

# 3. Welch's t-test (does not assume equal variances)
print("\n3. Welch's t-test (unequal-variance assumption):")
welch_t_stat, welch_p_value = stats.ttest_ind(face_extrema, place_extrema, equal_var=False)
print(f"   t statistic: {welch_t_stat:.4f}")
print(f"   p-value: {welch_p_value:.4f}")

# 4. Mann-Whitney U test (non-parametric)
print("\n4. Mann-Whitney U test (non-parametric):")
u_stat, u_p_value = stats.mannwhitneyu(face_extrema, place_extrema, alternative='two-sided')
print(f"   U statistic: {u_stat:.4f}")
print(f"   p-value: {u_p_value:.4f}")

# Check homogeneity of variance
print("\n=== Homogeneity-of-variance test ===")
levene_stat, levene_p = stats.levene(face_extrema, place_extrema)
print(f"Levene test: F={levene_stat:.4f}, p={levene_p:.4f}")
if levene_p < 0.05:
    print("  -> unequal variances (Welch's t-test recommended)")
else:
    print("  -> equal-variance assumption holds")

# Check normality
print("\n=== Normality test ===")
face_shapiro_stat, face_shapiro_p = stats.shapiro(face_extrema)
place_shapiro_stat, place_shapiro_p = stats.shapiro(place_extrema)

print(f"Face data Shapiro-Wilk: W={face_shapiro_stat:.4f}, p={face_shapiro_p:.4f}")
print(f"Place data Shapiro-Wilk: W={place_shapiro_stat:.4f}, p={place_shapiro_p:.4f}")

if face_shapiro_p < 0.05 or place_shapiro_p < 0.05:
    print("  -> at least one group is non-normal (non-parametric test recommended)")
else:
    print("  -> both groups are approximately normal")

## Concave & Convex
## Face versus Place

In [ ]:
# Extract the Extrema_Points data
face_extrema = inflection_face_Concave['Extrema_Points']
place_extrema = inflection_place_Convex['Extrema_Points']

print("=== Descriptive statistics ===")
print(f"Face (Concave) - n: {len(face_extrema)}")
print(f"  mean: {np.mean(face_extrema):.4f}")
print(f"  SD: {np.std(face_extrema, ddof=1):.4f}")
print(f"  median: {np.median(face_extrema):.4f}")

print(f"\nPlace (Convex) - n: {len(place_extrema)}")
print(f"  mean: {np.mean(place_extrema):.4f}")
print(f"  SD: {np.std(place_extrema, ddof=1):.4f}")
print(f"  median: {np.median(place_extrema):.4f}")

print("\n=== Statistical test results ===")

# 1. Permutation Test
print("1. Permutation Test:")
observed_diff, perm_p_value, perm_diffs = permutation_test(face_extrema, place_extrema)
print(f"   observed mean difference: {observed_diff:.4f}")
print(f"   p-value: {perm_p_value:.4f}")

# 3. Welch's t-test (does not assume equal variances)
print("\n3. Welch's t-test (unequal-variance assumption):")
welch_t_stat, welch_p_value = stats.ttest_ind(face_extrema, place_extrema, equal_var=False)
print(f"   t statistic: {welch_t_stat:.4f}")
print(f"   p-value: {welch_p_value:.4f}")

# 4. Mann-Whitney U test (non-parametric)
print("\n4. Mann-Whitney U test (non-parametric):")
u_stat, u_p_value = stats.mannwhitneyu(face_extrema, place_extrema, alternative='two-sided')
print(f"   U statistic: {u_stat:.4f}")
print(f"   p-value: {u_p_value:.4f}")

# Check homogeneity of variance
print("\n=== Homogeneity-of-variance test ===")
levene_stat, levene_p = stats.levene(face_extrema, place_extrema)
print(f"Levene test: F={levene_stat:.4f}, p={levene_p:.4f}")
if levene_p < 0.05:
    print("  -> unequal variances (Welch's t-test recommended)")
else:
    print("  -> equal-variance assumption holds")

# Check normality
print("\n=== Normality test ===")
face_shapiro_stat, face_shapiro_p = stats.shapiro(face_extrema)
place_shapiro_stat, place_shapiro_p = stats.shapiro(place_extrema)

print(f"Face data Shapiro-Wilk: W={face_shapiro_stat:.4f}, p={face_shapiro_p:.4f}")
print(f"Place data Shapiro-Wilk: W={place_shapiro_stat:.4f}, p={place_shapiro_p:.4f}")

if face_shapiro_p < 0.05 or place_shapiro_p < 0.05:
    print("  -> at least one group is non-normal (non-parametric test recommended)")
else:
    print("  -> both groups are approximately normal")

## Face concave versus Face convex

In [ ]:
# Extract the Extrema_Points data
face_extrema = inflection_face_Concave['Extrema_Points']
place_extrema = inflection_face_Convex['Extrema_Points']

print("=== Descriptive statistics ===")
print(f"Face (Concave) - n: {len(face_extrema)}")
print(f"  mean: {np.mean(face_extrema):.4f}")
print(f"  SD: {np.std(face_extrema, ddof=1):.4f}")
print(f"  median: {np.median(face_extrema):.4f}")

print(f"\nFace (Convex) - n: {len(place_extrema)}")
print(f"  mean: {np.mean(place_extrema):.4f}")
print(f"  SD: {np.std(place_extrema, ddof=1):.4f}")
print(f"  median: {np.median(place_extrema):.4f}")

print("\n=== Statistical test results ===")


print("Permutation Test:")
observed_diff, perm_p_value, perm_diffs = permutation_test(face_extrema, place_extrema)
print(f"   observed mean difference: {observed_diff:.4f}")
print(f"   p-value: {perm_p_value:.4f}")


## Place concave versus Place convex

In [ ]:
# Extract the Extrema_Points data
place_concave_extrema = inflection_place_Concave['Extrema_Points']
place_convex_extrema = inflection_place_Convex['Extrema_Points']

print("=== Descriptive statistics ===")
print(f"Place (Concave) - n: {len(place_concave_extrema)}")
print(f"  mean: {np.mean(place_concave_extrema):.4f}")
print(f"  SD: {np.std(place_concave_extrema, ddof=1):.4f}")
print(f"  median: {np.median(place_concave_extrema):.4f}")

print(f"\nPlace (Convex) - n: {len(place_convex_extrema)}")
print(f"  mean: {np.mean(place_convex_extrema):.4f}")
print(f"  SD: {np.std(place_convex_extrema, ddof=1):.4f}")
print(f"  median: {np.median(place_convex_extrema):.4f}")

print("\n=== Statistical test results ===")


print("Permutation Test:")
observed_diff, perm_p_value, perm_diffs = permutation_test(place_concave_extrema, place_convex_extrema)
print(f"   observed mean difference: {observed_diff:.4f}")
print(f"   p-value: {perm_p_value:.4f}")


## All Concave versus All Convex

Update: add several body and tool regions

In [ ]:
All_Concave = pd.concat([inflection_face_Concave, inflection_place_Concave, inflection_body_Concave, inflection_tool_Concave], ignore_index=True)
All_Convex = pd.concat([inflection_face_Convex, inflection_place_Convex, inflection_body_Convex, inflection_tool_Convex], ignore_index=True)


In [ ]:
All_Concave.dropna().count()

In [ ]:
All_Convex.dropna().count()

In [ ]:
# Extract the Extrema_Points data
all_concave_extrema = All_Concave['Extrema_Points']
all_convex_extrema = All_Convex['Extrema_Points']

print("=== Descriptive statistics ===")
print(f"All Concave - n: {len(all_concave_extrema)}")
print(f"  mean: {np.mean(all_concave_extrema):.4f}")
print(f"  SD: {np.std(all_concave_extrema, ddof=1):.4f}")
print(f"  median: {np.median(all_concave_extrema):.4f}")

print(f"\nAll Convex - n: {len(all_convex_extrema)}")
print(f"  mean: {np.mean(all_convex_extrema):.4f}")
print(f"  SD: {np.std(all_convex_extrema, ddof=1):.4f}")
print(f"  median: {np.median(all_convex_extrema):.4f}")

print("\n=== Statistical test results ===")

print("Permutation Test:")
observed_diff, perm_p_value, perm_diffs = permutation_test(all_concave_extrema, all_convex_extrema)
print(f"   observed mean difference: {observed_diff:.4f}")
print(f"   p-value: {perm_p_value:.4f}")


In [ ]:
inflection_place_Convex['Extrema_Points'].dropna().std()


In [ ]:
print(inflection_place_Concave['Extrema_Points'].dropna().count())
print(inflection_place_Concave['Extrema_Points'].dropna().std())
print(inflection_place_Concave['Extrema_Points'].dropna().mean())

In [ ]:
print(inflection_place_Convex['Extrema_Points'].dropna().count())
print(inflection_place_Convex['Extrema_Points'].dropna().std())
print(inflection_place_Convex['Extrema_Points'].dropna().mean())

In [ ]:
print(inflection_face_Convex['Extrema_Points'].dropna().count())
print(inflection_face_Convex['Extrema_Points'].dropna().std())
print(inflection_face_Convex['Extrema_Points'].dropna().mean())

In [ ]:
print(inflection_face_Concave['Extrema_Points'].dropna().count())
print(inflection_face_Concave['Extrema_Points'].dropna().std())
print(inflection_face_Concave['Extrema_Points'].dropna().mean())

In [ ]:
inflection_body_Convex['Extrema_Points'].dropna().count()

In [ ]:
inflection_body_Convex['Extrema_Points'].dropna().mean()

In [ ]:
print(inflection_tool_Concave['Extrema_Points'].dropna().count())
print(inflection_tool_Concave['Extrema_Points'].dropna().std())
print(inflection_tool_Concave['Extrema_Points'].dropna().mean())


In [ ]:
inflection_face_Concave

In [ ]:
face_place_concave = pd.concat([inflection_face_Concave, inflection_place_Concave], ignore_index=True)
face_place_convex = pd.concat([inflection_face_Convex, inflection_place_Convex], ignore_index=True)

In [ ]:
face_place_concave.dropna(inplace = True)
face_place_convex.dropna(inplace = True)

In [ ]:
len(face_place_concave)

In [ ]:
len(face_place_convex)

In [ ]:
face_place_concave['Extrema_Points'].mean()

In [ ]:
face_place_concave['Extrema_Points'].std()

In [ ]:
face_place_convex['Extrema_Points'].mean()

In [ ]:
face_place_convex['Extrema_Points'].std()